# 02 Text Preprocessing, Stemming vs Lemmatization & Subword Tokenization

**Scenario**: Enterprise Subword Tokenization & Stemming vs Lemmatization Benchmarks.

This notebook demonstrates comparing Porter Stemmer vs WordNet Lemmatizer, executing a custom 3-iteration BPE algorithm, and running BPE subword tokenization.

## Step 1: Stemming vs. Lemmatization Comparison

In [1]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["computing", "computers", "computational", "organization", "organizing", "better"]

print("Word".ljust(15), "Porter Stemmer".ljust(18), "WordNet Lemmatizer")
print("-"*50)
for w in words:
    stem = stemmer.stem(w)
    lemma = lemmatizer.lemmatize(w, pos='v' if w.endswith('ing') else 'n')
    print(w.ljust(15), stem.ljust(18), lemma)

Word            Porter Stemmer     WordNet Lemmatizer
--------------------------------------------------


computing       comput             compute
computers       comput             computer
computational   comput             computational
organization    organ              organization
organizing      organ              organize
better          better             better


## Step 2: Custom 3-Iteration Byte Pair Encoding (BPE) Algorithm

In [2]:
from collections import defaultdict

# Sample corpus dictionary with frequencies
corpus = {
    'c o s t </w>': 4,
    'c o s t s </w>': 2,
    'c o s t i n g </w>': 3
}

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

import re
current_vocab = corpus.copy()
print("=== BPE Iterative Merging (3 Iterations) ===")
for i in range(1, 4):
    pairs = get_stats(current_vocab)
    best_pair = max(pairs, key=pairs.get)
    current_vocab = merge_vocab(best_pair, current_vocab)
    print(f"Iteration {i}: Best Pair {best_pair} (freq={pairs[best_pair]}) -> New Vocab Sample:", list(current_vocab.keys())[0])

=== BPE Iterative Merging (3 Iterations) ===
Iteration 1: Best Pair ('c', 'o') (freq=9) -> New Vocab Sample: co s t </w>
Iteration 2: Best Pair ('co', 's') (freq=9) -> New Vocab Sample: cos t </w>
Iteration 3: Best Pair ('cos', 't') (freq=9) -> New Vocab Sample: cost </w>


## Step 3: Production BPE Subword Encoding with Tiktoken

In [3]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
sample_text = "PostgreSQL database connection timeout exception: postgresql_replica_timeout"
tokens = enc.encode(sample_text)
subwords = [enc.decode([t]) for t in tokens]

print("=== Tiktoken Production Subword Breakdown ===")
print("Original Text:", sample_text)
print("Token IDs:", tokens)
print("Decoded Subwords:", subwords)

=== Tiktoken Production Subword Breakdown ===
Original Text: PostgreSQL database connection timeout exception: postgresql_replica_timeout
Token IDs: [4226, 60896, 4729, 3717, 9829, 4788, 25, 60926, 1498, 26614, 15677, 21179]
Decoded Subwords: ['Post', 'greSQL', ' database', ' connection', ' timeout', ' exception', ':', ' postgres', 'ql', '_rep', 'lica', '_timeout']


## Output Explanation & Complexity Analysis

- **Porter Stemmer vs. WordNet Lemmatizer**: Stemmer chops suffixes blindly (`"comput"`), while Lemmatizer retrieves valid canonical lemmas (`"compute"`).
- **BPE Merge Loop**: 3 iterations merge frequent character pairs `['c', 'o', 's', 't', 'i', 'n', 'g'] \rightarrow ['c', 'o', 's', 't', 'i', 'n', 'g', 'st', 'cost']`.
- **BPE Training Time Complexity**: $O(K \cdot N \cdot L)$ where $K$ is merge iteration count.
- **BPE Inference Tokenization Time Complexity**: $O(L \log |V|)$ subword lookup time.
- **Space Complexity**: $O(|V| + S)$ memory footprint for vocabulary and merge rules table.
- **Component Denotations**:
  - $N$: Total number of documents.
  - $L$: Token length of sequence.
  - $|V|$: Target subword vocabulary size.
  - $K$: Number of merge iterations.
  - $S$: Number of learned subword merge rules.